# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/saadtalat111/flyrank/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one **(client_hash_id, content_hash_id, report_date)** observation — a single pseudonymized content item, for a single pseudonymized client, on a single day, taken from `fact_content_daily_performance`.

**Decision anchor:** for this contract I define a `decision_date` of **2026-03-31** — the end of the mid-panel month (`month=2026-03`) I'm required to iterate on, and far from the sealed test month (June 2026).

**Two windows, deliberately kept apart (this is the whole point of a future-window label):**
- **Feature window:** `report_date <= decision_date`. For this notebook's proof-of-concept queries I use just the `month=2026-03` partition as my feature window slice — a full production 90-day trailing window would span the `2026-01`, `2026-02`, and `2026-03` partitions together, which I note as a limitation in Section 4 rather than build here.
- **Label window:** `report_date > decision_date`, specifically the next 30 days, i.e. `month=2026-04`. This is strictly *after* the feature window — nothing from April is allowed to leak into a feature.

I verify the grain, and pull row counts + the real date span for `month=2026-03`, below.


In [1]:
# --- Connect to the FlyRank warehouse (Hugging Face, gated, read-only) ---
# Docs: https://huggingface.co/datasets/FlyRank/internship-warehouse
import duckdb

from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")  # never paste the token itself in a cell — this repo is public

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"  # mid-panel month — NEVER the _sample (that's June 2026, the sealed test month)
LABEL_MONTH = "2026-04"  # the following month — used ONLY to build the label, never as a feature

fact_path = f"{WAREHOUSE}/fact_content_daily_performance/month={MONTH}/*.parquet"
fact_path_label = f"{WAREHOUSE}/fact_content_daily_performance/month={LABEL_MONTH}/*.parquet"

# Look at real columns before assuming any exist — don't guess a schema.
schema = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{fact_path}')").df()
print(schema.to_string(index=False))


             column_name column_type null  key default extra
             report_date        DATE  YES None    None  None
          client_hash_id     VARCHAR  YES None    None  None
         content_hash_id     VARCHAR  YES None    None  None
          client_has_gsc     BOOLEAN  YES None    None  None
          client_has_ga4     BOOLEAN  YES None    None  None
      gsc_data_available     BOOLEAN  YES None    None  None
      ga4_data_available     BOOLEAN  YES None    None  None
         gsc_impressions      BIGINT  YES None    None  None
              gsc_clicks      BIGINT  YES None    None  None
        gsc_sum_position      BIGINT  YES None    None  None
        gsc_avg_position      DOUBLE  YES None    None  None
           ga4_pageviews      BIGINT  YES None    None  None
            ga4_sessions      BIGINT  YES None    None  None
               ga4_users      BIGINT  YES None    None  None
    ga4_engaged_sessions      BIGINT  YES None    None  None
ga4_total_engagement_sec

In [2]:
# --- Verification query 1 of 3: GRAIN ---
# If the grain really is (report_date, client_hash_id, content_hash_id), this returns ZERO rows.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{fact_path}')
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f"Rows that violate the stated grain: {len(grain_check)}")
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows that violate the stated grain: 0


,report_date,client_hash_id,content_hash_id,c


In [3]:
# --- Verification query 2 of 3: ROW COUNT + DATE SPAN for my slice ---
counts = con.sql(f"""
    SELECT COUNT(*)                         AS n_rows,
           COUNT(DISTINCT client_hash_id)    AS n_clients,
           COUNT(DISTINCT content_hash_id)   AS n_content_items,
           MIN(report_date)                 AS min_date,
           MAX(report_date)                 AS max_date
    FROM read_parquet('{fact_path}')
""").df()

counts



,n_rows,n_clients,n_content_items,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

I ran `DESCRIBE` on the real table before writing this — this is the actual schema, not a guess from the docs (the docs turned out to be a bit stale: it's `client_hash_id`/`content_hash_id`, not `client_id`/`content_id`).

| Field(s) | Bucket | Why |
|---|---|---|
| `report_date`, `month` | **Context** | Define the feature/label windows and the partition; never model inputs themselves. |
| `client_hash_id`, `content_hash_id` | **Context** | Pseudonyms — grouping/joining/splitting only (client-holdout splits), never features. |
| `client_has_gsc`, `client_has_ga4` | **Context** | Client-level flags that repeat on every row for that client — read with `ANY_VALUE()`/`MAX()`, never summed (the "repeated context column" trap from the contract skill). Used for filtering, not as a learned signal. |
| `gsc_data_available`, `ga4_data_available` | **Context** | Row-level availability flags — decide which rows can safely use GSC/GA4 fields at all; not signals themselves. |
| `gsc_impressions`, `gsc_clicks`, `gsc_avg_position` | **Feature** | Fully inside the feature window (`report_date <= decision_date`) — knowable at decision time. `gsc_avg_position = 0` likely means "no data," not rank zero (the starter-CSV gotcha); I check this convention holds here before trusting it, in the missingness cell below. |
| `gsc_sum_position` | **Excluded** | Redundant raw component of `gsc_avg_position` (`sum / impressions ≈ avg`) — including both would double-count the same signal. |
| `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec` | **Feature (candidate)** | Knowable within the feature window, gated on `ga4_data_available`. Not all five are in my chosen 5-feature set below, but any of them is fair game later. |
| `sessions_organic`, `sessions_direct`, `sessions_referral`, `sessions_social`, `sessions_paid`, `sessions_ai` | **Feature (candidate)** | Traffic-source breakdown, knowable within the feature window. Not used in my 5 features yet — parked for later lane work, not excluded outright. |
| `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` | **Excluded** | Per Week 1, AI-referral signal is sparse and out of scope for Lane 2's core question; also, a session referred *from* an AI tool is not the same as being *cited by* one — no AI-citation claim should ever be drawn from these. |
| `scroll_events` | **Feature (candidate)** | Engagement-depth signal, knowable within the window — parked for now, not used in my 5. |
| `declining_next30` (derived, not a warehouse column) | **Label** | Built by comparing March (feature window) vs April (label window) `gsc_avg_position` for the same `content_hash_id`. Computed entirely from data outside the feature window — must never appear as a feature. Formula is below, next to the trap experiment. |

**Deliberately excluded, proven not just promised:** anything computed from `declining_next30` or from April data is excluded from the feature set by construction — the trap experiment below builds one on purpose specifically to show what that looks like when it goes wrong.


In [4]:
# Missingness spot-check on the one feature I've committed to above (gsc_avg_position),
# split by whether GA4 history exists for that row - patterns often follow a flag like this.
missingness = con.sql(f"""
    SELECT
        ga4_data_available,
        COUNT(*) AS n_rows,
        ROUND(100.0 * AVG(CASE WHEN gsc_avg_position = 0 THEN 1.0 ELSE 0 END), 1) AS pct_gsc_position_zero
    FROM read_parquet('{fact_path}')
    GROUP BY ga4_data_available
""").df()

missingness



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,ga4_data_available,n_rows,pct_gsc_position_zero
0,<NA>,3018741,2.7
1,False,6408671,1.2
2,True,413966,0.8


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Grain and row-count/date-span are already verified in Section 1 (queries 1–2). The third required query — **availability** — is below, followed by the five-feature frame and the leakage trap.


In [5]:
# --- Verification query 3 of 3: AVAILABILITY ---
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM read_parquet('{fact_path}')
""").df()

availability

,total_rows,ga4_available_rows,pct_available
0,9841378,413966.0,4.2


### Five features, max (from the feature window only)

All five are aggregated per `content_hash_id` over `month=2026-03` only — nothing from April touches these.

1. **`avg_gsc_position_mar`** — mean `gsc_avg_position` over March (excluding `0` = no-data rows). *Knowable at the decision moment because it only uses `report_date <= decision_date`.*
2. **`total_gsc_impressions_mar`** — sum of `gsc_impressions` over March, a demand signal. *Knowable because it only sums days already observed by the decision date.*
3. **`gsc_ctr_mar`** — `SUM(gsc_clicks) / SUM(gsc_impressions)` over March, a click-efficiency signal. *Knowable because both halves come entirely from the feature window.*
4. **`ga4_engagement_rate_mar`** — `SUM(ga4_engaged_sessions) / SUM(ga4_sessions)` over March, gated on `ga4_data_available`. *Knowable because it's computed purely within the feature window, and the gate keeps zero-filled non-available rows from faking a real rate.*
5. **`n_days_observed_mar`** — row count for this `content_hash_id` in March (panel coverage/attendance). *Knowable because it only reflects days already in the warehouse before the decision date.*


In [6]:
# --- Build the five-feature frame (feature window only: March) ---
features_mar = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_gsc_position_mar,
        SUM(gsc_impressions)                                          AS total_gsc_impressions_mar,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0)       AS gsc_ctr_mar,
        SUM(CASE WHEN ga4_data_available THEN ga4_engaged_sessions END) * 1.0
            / NULLIF(SUM(CASE WHEN ga4_data_available THEN ga4_sessions END), 0) AS ga4_engagement_rate_mar,
        COUNT(*)                                                      AS n_days_observed_mar
    FROM read_parquet('{fact_path}')
    GROUP BY content_hash_id
""").df()

print(f"Feature frame: {len(features_mar):,} content items")
features_mar.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 331,437 content items


,content_hash_id,avg_gsc_position_mar,total_gsc_impressions_mar,gsc_ctr_mar,ga4_engagement_rate_mar,n_days_observed_mar
0,content_7a105f548d9c6916,7.209549,6523.0,0.001073,0.0,31
1,content_a3ea9792f793ec72,3.307255,453.0,0.000000,NaN,31
2,content_36c36abc7650d7af,6.724039,5630.0,0.001066,0.0,31
3,content_a7da352b73b02668,7.244844,4944.0,0.002629,0.0,31
4,content_f39be42b42a4e8f6,23.314103,42.0,0.000000,0.0,31


### The trap: one label-derived column, on purpose

**Honest label (from April, the label window):** `declining_next30 = 1` if a content item's average `gsc_avg_position` gets *worse* (higher number = lower rank) from March to April, else `0`.

**Honest "quick score":** a simple rule using only March features — flag a content item as at-risk if its `gsc_ctr_mar` is below the median (low click efficiency now, as a leading indicator). No April data touches this rule at all.

**The trap:** I then add `avg_gsc_position_apr` itself — pulled straight from the label window — as if it were a "feature," and recompute the same quick score using it directly. Watch what happens to accuracy.


In [7]:
# --- Build the honest label from the LABEL window (April) ---
label_apr = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_gsc_position_apr
    FROM read_parquet('{fact_path_label}')
    GROUP BY content_hash_id
""").df()

panel = features_mar.merge(label_apr, on="content_hash_id", how="inner").dropna(
    subset=["avg_gsc_position_mar", "avg_gsc_position_apr", "gsc_ctr_mar"]
)
panel["declining_next30"] = (panel["avg_gsc_position_apr"] > panel["avg_gsc_position_mar"]).astype(int)

print(f"Content items with data in both months: {len(panel):,}")
print(f"Share declining next 30 days: {panel['declining_next30'].mean():.1%}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Content items with data in both months: 157,191
Share declining next 30 days: 64.3%


In [8]:
# --- Honest quick score: March-only rule, no future info ---
ctr_median = panel["gsc_ctr_mar"].median()
panel["predicted_honest"] = (panel["gsc_ctr_mar"] < ctr_median).astype(int)
honest_accuracy = (panel["predicted_honest"] == panel["declining_next30"]).mean()

print(f"Honest quick score (March-only rule): {honest_accuracy:.1%}")


Honest quick score (March-only rule): 35.7%


In [9]:
# --- THE TRAP: add a label-derived column and watch the score jump ---
# avg_gsc_position_apr comes straight from the label window - this should NEVER be a feature.
panel["predicted_leaked"] = (panel["avg_gsc_position_apr"] > panel["avg_gsc_position_mar"]).astype(int)
leaked_accuracy = (panel["predicted_leaked"] == panel["declining_next30"]).mean()

print(f"Leaked quick score (using April data as a 'feature'): {leaked_accuracy:.1%}")
print(f"Honest quick score, for comparison:                    {honest_accuracy:.1%}")
print()
print("The leaked score is trivially near-perfect because the 'feature' IS the label's own formula.")
print("This is exactly the shape of leakage to watch for once real feature building starts.")


Leaked quick score (using April data as a 'feature'): 100.0%
Honest quick score, for comparison:                    35.7%

The leaked score is trivially near-perfect because the 'feature' IS the label's own formula.
This is exactly the shape of leakage to watch for once real feature building starts.


In [10]:
# --- Delete the leaked column, keep only the honest number ---
panel = panel.drop(columns=["predicted_leaked"])
print("Leaked column removed. Kept feature columns:")
print([c for c in panel.columns if c not in ("declining_next30", "predicted_honest", "avg_gsc_position_apr")])


Leaked column removed. Kept feature columns:
['content_hash_id', 'avg_gsc_position_mar', 'total_gsc_impressions_mar', 'gsc_ctr_mar', 'ga4_engagement_rate_mar', 'n_days_observed_mar']


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **This is one month's worth of feature data, not a real 90-day trailing window.** A production version needs to join `2026-01` + `2026-02` + `2026-03` partitions together before computing March-anchored 90-day features — I didn't build that join here, so today's features are a proof of concept, not the final pipeline.
- **GA4 data is sparse, and it's tri-state, not boolean.** `ga4_data_available` actually takes three values in practice: `True` (413,966 rows, 4.2%), `False` (6,408,671 rows), and **null** (3,018,741 rows — 30.7% of March). I assumed a clean True/False split before checking; the null group needs its own decision (treat as "unknown," not as "False") before any GA4 feature ships. `ga4_engagement_rate_mar` is `0.0` or `NaN` for most content items as a direct result — a blind `fillna(0)` on it would fabricate a "zero engagement" signal for the ~96% of rows that simply never had GA4 data at all.
- **Client access is unbalanced across 5 real categories, not a date range.** I assumed `dim_clients` had `gsc_data_start`/`ga4_data_start` columns before checking — it doesn't. Running the query below on all 104 clients gives: `gsc_and_ga4` (53), `no_search_or_analytics_access` (26), `gsc_only` (14), `source_only_missing_client_dimension` (10), `ga4_only` (1). Any client-level join needs to filter on `access_profile` first, or 26+ clients with zero search/analytics access will silently pollute the panel.
- **The `_sample` table (June 2026) is the sealed test month.** I never touch it for label logic — only the final honest evaluation, once, at the very end.
- **This data supports decision-support claims only** — associations, not causal claims about whether refreshing a page actually improves it (that needs an experiment this data can't run).


In [11]:
# The `dim_clients` path I first guessed (dim_clients/**/*.parquet) 404'd - rather than guess
# again, discover the real path: try common layouts, and fall back to listing the warehouse
# if none of them work. "Search before assuming" applies to file paths too, not just columns.
candidate_paths = [
    f"{WAREHOUSE}/dim_clients.parquet",
    f"{WAREHOUSE}/dim_clients/*.parquet",
    f"{WAREHOUSE}/data/dim_clients.parquet",
    f"{WAREHOUSE}/data/dim_clients/*.parquet",
    f"{WAREHOUSE}/data/dim_clients/train.parquet",
]

dim_clients_path = None
for p in candidate_paths:
    try:
        n = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{p}')").df()["n"][0]
        print(f"WORKS: {p}  (n={n} rows)")
        dim_clients_path = p
        break
    except Exception:
        print(f"tried: {p} -> not found")

if dim_clients_path is None:
    print("\nNone of the guesses worked - listing the warehouse to find the real path:")
    listing = con.sql(f"SELECT * FROM glob('{WAREHOUSE}/**')").df()
    hits = listing[listing.iloc[:, 0].str.contains("client", case=False, na=False)]
    print(hits.to_string(index=False))
    print("\nCopy the correct path from above into dim_clients_path below and re-run.")


WORKS: hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet  (n=104 rows)


In [12]:
# Confirm the panel-access warning with real numbers
assert dim_clients_path is not None, "Set dim_clients_path from the discovery cell above first"

clients = con.sql(f"""
    SELECT
        access_profile,
        COUNT(*) AS n_clients,
        SUM(CASE WHEN is_active THEN 1 ELSE 0 END) AS n_active
    FROM read_parquet('{dim_clients_path}')
    GROUP BY access_profile
    ORDER BY n_clients DESC
""").df()

clients


,access_profile,n_clients,n_active
0,gsc_and_ga4,53,41.0
1,no_search_or_analytics_access,26,18.0
2,gsc_only,14,14.0
3,source_only_missing_client_dimension,10,0.0
4,ga4_only,1,1.0


**Named limitation, not a generic one:** all 10 clients in `source_only_missing_client_dimension` have `n_active = 0` — every client in that bucket, with zero exceptions. That's not "some clients lack access"; it's a specific data-quality gap where the dimension record itself looks incomplete (present in the fact table as a source, but missing proper client-dimension data). Any client-level analysis needs to either exclude this bucket explicitly or investigate it separately — silently including it would mix genuinely-inactive clients with clients whose dimension row is simply broken.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.